<img src="https://fedesarrollo.org.co/logo-Fede.png" alt="Fedesarrollo" style="float: right; width: 300px; height: auto;">
<img src="https://upload.wikimedia.org/wikipedia/commons/f/f9/Logo_BID_Espa%C3%B1ol.png" alt="Banco Interamericano de Desarrollo" style="float: right; width: 220px; height: auto;">

# Ventaja en medidas no arancelarias

Editor: Juan Diego Heredia Niño - jheredia@fedesarrollo.org.co

Septiembre 2026

### Importar librerías

La siguiente celda marca el inicio de un nuevo bloque de trabajo independiente:

**Librerías utilizadas:**
- `pandas` y `numpy`: para manipulación y análisis de datos estructurados.

In [1]:
import pandas as pd
import numpy as np

### Configuración de rutas

Carga las rutas del proyecto y las opciones globales desde `config.py`, instalado como
paquete editable en la raíz del proyecto.

In [2]:
from config import temp, tables

### Lectura de datos

Carga las medidas no arancelarias consolidadas y la cartera con tamaño de mercado.

In [3]:
df_ntms = pd.read_parquet(temp / 'df_ntms.parquet')
df_products = pd.read_parquet(temp / 'df_products_mercado.parquet')

carpeta_tariffs = tables / 'tariffs'
carpeta_tariffs.mkdir(parents=True, exist_ok=True)

### Ventaja no arancelaria

Compara el número de medidas que cada país impone a Colombia con el promedio que impone al
resto de socios para el mismo código arancelario. La ventaja es uno cuando Colombia enfrenta
un número de medidas menor o igual al promedio.

In [4]:
df_ntms_agg = df_ntms.groupby(['Reporter Name', 'Partner Name','HS'])['NTM code'].count().reset_index()
df_ntms_col = df_ntms_agg.query("`Partner Name` == 'Colombia'")
df_ntms_agg = df_ntms_agg.query("`Partner Name` != 'Colombia'").groupby(['Reporter Name','HS'])['NTM code'].mean().reset_index()

df_ntms_col = df_ntms_agg.merge(df_ntms_col, on=['Reporter Name','HS'], how='outer', suffixes=('_world','_col')).drop_duplicates().reset_index(drop=True)

df_ntms_col['Partner Name'] = df_ntms_col['Partner Name'].fillna('Colombia')
df_ntms_col['NTM code_world'] = df_ntms_col['NTM code_world'].fillna(0)
df_ntms_col['NTM code_col'] = df_ntms_col['NTM code_col'].fillna(0)

df_ntms_col['NTM world col ratio'] = (1 + df_ntms_col['NTM code_world']) / (1 + df_ntms_col['NTM code_col'])
df_ntms_col['Ventaja no Arancelaria'] = np.where(df_ntms_col['NTM world col ratio'] >= 1, 1, 0)

df_ntms_col = df_ntms_col.groupby(['HS'])['Ventaja no Arancelaria'].mean().reset_index()

### Imputación a la cartera

Asigna la ventaja por código de seis dígitos, la completa con la de cuatro y, en su defecto,
con el promedio del capítulo de dos dígitos.

In [5]:
df_products['HS4'] = df_products['HS6'].str[:4]
df_products['HS2'] = df_products['HS6'].str[:2]
df_products = df_products.merge(df_ntms_col, left_on=['HS6'], right_on=['HS'], how='left').merge(df_ntms_col, left_on=['HS4'], right_on=['HS'], how='left')
df_products['Ventaja no Arancelaria'] = df_products['Ventaja no Arancelaria_x'].fillna(df_products['Ventaja no Arancelaria_y'])
df_products = df_products.drop(columns=['HS4', 'HS_x', 'HS_y', 'Ventaja no Arancelaria_x', 'Ventaja no Arancelaria_y'])
df_products = df_products.merge(df_products[['HS2','Ventaja no Arancelaria']].groupby('HS2')['Ventaja no Arancelaria'].mean().reset_index(), on='HS2', how='left', suffixes=('', '_HS2'))
df_products['Ventaja no Arancelaria'] = df_products['Ventaja no Arancelaria'].fillna(df_products['Ventaja no Arancelaria_HS2'])
df_products = df_products.drop(columns=['Ventaja no Arancelaria_HS2','HS2','Fuente','Categoría'])

df_products.to_excel(carpeta_tariffs / 'df_ntms.xlsx', index=False)